### Scenario 1: A single data scientist participating in an ML competition

MLflow setup:

- Tracking server: no
- Backend store: local filesystem
- Artifacts store: local filesystem

In [1]:
import mlflow
from mlflow.tracking import MlflowClient

print(f"tracking URI: '{mlflow.get_tracking_uri()}'") # it asumes the uri is in the same folder at this notebook.
client = MlflowClient(tracking_uri=mlflow.get_tracking_uri())

mlflow.search_experiments() # if mlops folder does not exists it creates it.

tracking URI: 'file:///home/yezergm/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlruns'


[<Experiment: artifact_location='file:///home/yezergm/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlruns/0', creation_time=1744712671523, experiment_id='0', last_update_time=1744712671523, lifecycle_stage='active', name='Default', tags={}>]

##### Creating an experiment and logging a new run

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from mlflow.tracking import MlflowClient
import mlflow

client = MlflowClient(tracking_uri=mlflow.get_tracking_uri())

client.create_experiment("experiment-01") # the experiment_id value coulb be not incremental.
mlflow.search_experiments()


[<Experiment: artifact_location='file:///home/yezergm/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlruns/819886757721026669', creation_time=1744712686350, experiment_id='819886757721026669', last_update_time=1744712686350, lifecycle_stage='active', name='experiment-01', tags={}>,
 <Experiment: artifact_location='file:///home/yezergm/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlruns/0', creation_time=1744712671523, experiment_id='0', last_update_time=1744712671523, lifecycle_stage='active', name='Default', tags={}>]

In [3]:
# In this case the models has not been registered, but is not needed beacause, I'm doing tests in local,
#  I don't need to register the models.
mlflow.set_experiment("experiment-01") 

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2025/04/15 10:31:09 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


default artifacts URI: 'file:///home/yezergm/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlruns/819886757721026669/6687af25590f493cb78565da4d8b7c4d/artifacts'


In [13]:
y_pred

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

##### access to models

In [8]:
# now threre are not registered models, but I want to access to the registry
from mlflow.tracking import MlflowClient
from mlflow.exceptions import MlflowException

client = MlflowClient(tracking_uri=mlflow.get_tracking_uri())

try:
    client.get_registered_model(name="experiment-01")
except MlflowException:
    print("It's not possible to access the model registry :(" + \
          "\nbut is possible from command line")

It's not possible to access the model registry :(


cli, in mlruns folder to access to the ui
$ mlflow ui

##### retrieve a model

In [20]:
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.exceptions import MlflowException
import numpy as np  

client = MlflowClient(tracking_uri=mlflow.get_tracking_uri())

run_id = "6687af25590f493cb78565da4d8b7c4d"
run_info =client.get_run(run_id)
artifact_uri = run_info.info.artifact_uri

# Load the model from the artifact URI
model_uri = f"{artifact_uri}/models"
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Make a prediction.
X, y = load_iris(return_X_y=True)
predictions = loaded_model.predict(X)

# Compare arrays using numpy's array_equal
np.array_equal(predictions, y_pred)

print(accuracy_score(y, predictions))


0.96
